# 04 — Deteksi Tepi, Gradien Citra, dan Ketelitian Lokalisasi

### Dari perubahan intensitas menuju batas geometris yang dapat diukur

**Pertanyaan utama:** apakah garis yang terlihat tajam pada citra selalu menunjukkan batas objek yang tepat? Ketika gradien diperkuat, apakah ketelitian pengukuran posisi tepi meningkat, atau noise justru semakin dominan?

Notebook ini mempertemukan kalkulus diskret, pemrosesan sinyal, model fisika pencitraan, dan metrologi. Kita tidak hanya menampilkan peta tepi: kita akan menurunkan operator diferensial, menguji skala deteksi, menjelaskan *Canny edge detector*, menyambungkan *edge spread function* (ESF) dengan PSF/MTF, lalu mengukur batas material dan ketidakpastiannya secara kuantitatif.

**Prasyarat:** Notebook 01 (model sensor, sampling, kuantisasi), 02 (histogram, kontras), dan 03 (filtering, konvolusi, PSF/OTF); kalkulus dasar, aljabar linear, probabilitas, dan NumPy.

**Alokasi yang disarankan:** 3–4 sesi kuliah/praktikum serta satu studi kasus mandiri. **Komputasi:** CPU, tanpa API berbayar dan tanpa pengunduhan dataset tambahan.

**Menjalankan setelah commit:** [Buka notebook di Google Colab](https://colab.research.google.com/github/iwanconysetiadi/ImageProcessing/blob/main/notebooks/04_tepi.ipynb).

> **Provenance:** citra `camera` dan `coins` merupakan data contoh `scikit-image`; seluruh eksperimen batas material, penentuan posisi tepi, foton, elektron, dan kalibrasi skala dalam studi kasus merupakan **simulasi sintetis**, bukan hasil kalibrasi atau eksperimen sensor fisik. Skala mm/piksel yang digunakan hanya merupakan asumsi demonstrasi.

## Peta kompetensi dan pertanyaan pemandu

Setelah menyelesaikan notebook, pembaca mampu:

1. Membedakan batas material, perubahan iluminasi, tekstur, dan tepi yang terdeteksi secara algoritmik.
2. Menurunkan aproksimasi beda hingga untuk turunan spasial, memahami respons frekuensinya, dan menjelaskan penguatan noise.
3. Menginterpretasikan gradien sebagai vektor: magnitudo, arah normal tepi, dan konvensi koordinat baris–kolom.
4. Menjelaskan dasar operator Prewitt, Sobel, Scharr, serta keterbatasan anisotropi dan normalisasi kernel.
5. Menurunkan hubungan *Gaussian smoothing*, turunan Gaussian, dan skala lokalisasi.
6. Membedakan ESF, LSF, PSF, OTF, MTF dan kondisi identifikasi masing-masing.
7. Menjelaskan secara konseptual *non-maximum suppression* dan *hysteresis thresholding* pada Canny.
8. Membedakan respons turunan pertama dan kedua, serta menguji perilaku Laplacian of Gaussian.
9. Mengestimasi posisi tepi subpiksel, memeriksa bias dan presisi, dan mengukur ketidakpastian.
10. Mengembangkan evaluasi *held-out* tanpa *data leakage* dan membandingkan galat geometris, keberhasilan deteksi, serta biaya komputasi.

**Pertanyaan pemandu:** Apa arti sebuah piksel tepi bila citra direkam oleh sensor ber-PSF lebar? Mengapa tepi tipis dari Canny tidak otomatis memberi posisi fisik yang paling akurat? Kapan penghalusan meningkatkan presisi, tetapi memperbesar bias?

## 1. Kontrak numerik, definisi koordinat, dan lingkungan

Citra dinyatakan sebagai $I[m,n]$, dengan $m$ indeks **baris** yang bertambah ke bawah dan $n$ indeks **kolom** yang bertambah ke kanan. Jika piksel mempunyai *pitch* $p_y$ dan $p_x$, koordinat fisik dapat ditulis $y=m p_y$ dan $x=n p_x$, setelah titik asal dan orientasi ditentukan melalui kalibrasi.

Pada array NumPy, `axis=0` berarti perubahan antarbari​s (arah vertikal), sedangkan `axis=1` berarti perubahan antarkolom (arah horizontal). Untuk citra monokrom yang dinormalisasi ke `[0,1]`, nilai tersebut merupakan **nilai kode yang telah diskalakan**, bukan otomatis radiansi atau reflektansi.

**Aturan eksperimen:** tetapkan `seed`; jangan mengubah *ground truth* menggunakan hasil prediksi; tampilkan `vmin`/`vmax` yang konsisten; bedakan data sintetis, citra contoh, dan data ukur fisik.

In [ ]:
import platform, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import ndimage, special
from scipy.integrate import trapezoid
from skimage import data, filters, feature, morphology, measure
from skimage.util import img_as_float
from IPython.display import display

SEED = 202604
rng = np.random.default_rng(SEED)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': False, 'font.size': 10})

def tampil(ax, im, judul, *, vmin=0.0, vmax=1.0, cmap='gray'):
    ax.imshow(im, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(judul)
    ax.axis('off')

camera = img_as_float(data.camera())[96:352, 96:352]
print('Python:', platform.python_version(), '| NumPy:', np.__version__)
print('Skala contoh citra:', camera.shape, '| nilai minimum/maksimum:',
      np.round([camera.min(), camera.max()], 3))
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
tampil(ax[0], camera, 'Citra contoh (scikit-image)')
ax[1].plot(camera[120, :], lw=1.2)
ax[1].set(xlabel='Kolom n (piksel)', ylabel='Nilai kode ternormalisasi',
          title='Profil intensitas pada satu baris')
ax[1].grid(alpha=.25)
plt.tight_layout(); plt.show()

## 2. Apa yang sebenarnya disebut tepi?

Istilah *tepi* memiliki sedikitnya tiga arti yang **tidak identik**:

1. **Batas fisik:** diskontinuitas atau perubahan sifat material, geometri, kedalaman, atau reflektansi.
2. **Transisi citra:** perubahan intensitas akibat gabungan objek, pencahayaan, PSF, sensor, sampling, dan proses digital.
3. **Tepi algoritmik:** piksel atau kurva yang lolos suatu kriteria deteksi, misalnya magnitudo gradien melebihi ambang dan merupakan maksimum lokal.

Batas fisik tidak selalu menghasilkan transisi kuat; bayangan atau pantulan dapat membentuk transisi kuat tanpa batas objek. Model sederhana untuk batas 1D ialah langkah ideal

$$
f(x)=a+b\,H(x-x_0),
$$

dengan $H$ fungsi Heaviside, $x_0$ posisi batas, $a$ latar, dan $b$ kontras bertanda. Jika PSF efektif $h$ bekerja dalam domain linier, profil terukur menjadi $g=h*f+\eta$. Turunannya memenuhi, secara distribusional dan untuk PSF yang memenuhi syarat,

$$
\frac{dg}{dx}(x)=b\,h(x-x_0)+\frac{d\eta}{dx}(x).
$$

Artinya, **lebar puncak turunan** membawa informasi tentang blur; sedangkan pusat puncaknya hanya menyatakan posisi batas dengan tepat bila model PSF dan akuisisinya tidak menimbulkan pergeseran sistematis.

In [ ]:
# Langkah ideal, ramp, garis tipis, dan bayangan memberi "tepi" berbeda
x = np.arange(220, dtype=float)
signals = {
    'Langkah material': .24 + .53*(x >= 110),
    'Ramp kontinu': .22 + .54*np.clip((x-85)/55, 0, 1),
    'Garis terang tipis': .25 + .58*np.exp(-.5*((x-110)/3.0)**2),
    'Bayangan (bukan batas material)': .75 - .42*special.ndtr((x-110)/6.0),
}
fig, axs = plt.subplots(2, 2, figsize=(11, 6.0))
for ax, (label, s) in zip(axs.ravel(), signals.items()):
    ax.plot(x, s, label='Intensitas', lw=2)
    ax.plot(x, np.gradient(s), label='Turunan beda tengah', lw=1.2)
    ax.axvline(110, ls=':', color='k', alpha=.4)
    ax.set(xlabel='Kolom (piksel)', ylabel='Sinyal / turunan', title=label)
    ax.legend(fontsize=8); ax.grid(alpha=.2)
plt.tight_layout(); plt.show()

## 3. Dari turunan kontinu ke beda hingga diskret

Turunan kontinu menggambarkan laju perubahan fungsi. Karena kamera hanya memberi sampel pada grid, kita harus memperkirakannya menggunakan beda hingga. Dengan jarak sampel $\Delta x$, turunan tengah ialah

$$
\left.\frac{\partial f}{\partial x}\right|_{x_n}
\approx\frac{f(x_n+\Delta x)-f(x_n-\Delta x)}{2\Delta x},
$$

dengan galat pemotongan $O(\Delta x^2)$ untuk fungsi yang cukup mulus. Turunan maju $[f(x+\Delta x)-f(x)]/\Delta x$ memiliki galat $O(\Delta x)$ dan letak efektif pada segmen berbeda.

Pada grid piksel dengan $\Delta x=1$ piksel, kernel korelasi $[-1/2,\;0,\;1/2]$ menghasilkan turunan tengah. Untuk sinyal diskret $f[n]=e^{i\omega n}$, operator ini memiliki respons

$$
D(\omega)=i\sin\omega,
$$

sedangkan turunan ideal pada sinyal kontinu adalah $i\omega$. Keduanya hanya sebanding untuk $|\omega|\ll 1$. Pada frekuensi Nyquist $\omega=\pi$, operator beda tengah menghasilkan respons **nol**, bukan respons ideal. Karena itu, diferensiasi diskret tidak sekadar "mengambil selisih piksel" tanpa konsekuensi spektral.

Jika noise piksel independen dengan varians $\sigma_\eta^2$, turunan tengah memiliki

$$
\operatorname{Var}\!\left(\frac{\eta[n+1]-\eta[n-1]}{2\Delta x}\right)
=\frac{\sigma_\eta^2}{2\Delta x^2}.
$$

Formula varians tersebut gagal bila noise antarpiksel berkorelasi; suku kovarians harus disertakan.

In [ ]:
# Bandingkan respons frekuensi dan penguatan noise untuk beda hingga
w = np.linspace(0, np.pi, 500)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(w/np.pi, w, label='Turunan ideal |ω|')
ax[0].plot(w/np.pi, np.abs(np.sin(w)), label='Beda tengah |sin ω|')
ax[0].plot(w/np.pi, 2*np.abs(np.sin(w/2)), label='Beda maju |2 sin(ω/2)|')
ax[0].set(xlabel='Frekuensi / frekuensi Nyquist', ylabel='Magnitudo respons (sampel⁻¹)',
          title='Bias frekuensi turunan diskret')
ax[0].legend(); ax[0].grid(alpha=.2)
noise_std = .04
samples = rng.normal(0, noise_std, size=(3000, 200))
dcen = (samples[:, 2:] - samples[:, :-2])/2
pred_var = noise_std**2/2
sim_var = np.var(dcen, ddof=1)
print(f'Varians beda tengah: teori {pred_var:.6f}, simulasi {sim_var:.6f}')
ax[1].hist(dcen.ravel(), bins=65, density=True, color='0.6', label='Noise setelah diferensiasi')
ax[1].set(xlabel='Gradien kode/piksel', ylabel='Densitas', title='Distribusi galat gradien')
ax[1].grid(alpha=.2)
plt.tight_layout(); plt.show()
assert np.isclose(sim_var, pred_var, rtol=.07)

## 4. Gradien dua dimensi dan konvensi arah

Gradien citra kontinu pada sistem koordinat $x$ (ke kanan) dan $y$ (ke bawah) adalah vektor

$$
\nabla I=\begin{bmatrix}I_x\\ I_y\end{bmatrix},\qquad
G=\sqrt{I_x^2+I_y^2},\qquad
\theta=\operatorname{atan2}(I_y,I_x).
$$

$G$ menyatakan besar perubahan lokal; $\theta$ adalah **arah normal terhadap kontur isointensitas**, bukan arah sepanjang garis tepi. Arah tangent kontur dapat diambil sebagai $\theta+\pi/2$. Konvensi sudut berubah jika koordinat $y$ didefinisikan positif ke atas, seperti pada geometri Kartesius.

Jika pitch piksel tidak isotropik, gunakan $I_x\approx \Delta_n I/p_x$ dan $I_y\approx \Delta_m I/p_y$. Gradien dalam kode/piksel **tidak** dapat langsung dibandingkan dengan gradien kode/mm tanpa konversi. Pada tepi dengan kontras negatif, arah normal berbalik $180^\circ$, tetapi garis batas fisiknya tetap sama.

In [ ]:
# Buktikan arah normal dan akibat anisotropi skala spasial
H, W = 135, 150
yy, xx = np.mgrid[:H, :W]
phantom = .15 + .68*special.ndtr((xx - (.42*yy + 40))/1.4)
gy, gx = np.gradient(phantom)
mag = np.hypot(gx, gy)
ang = np.arctan2(gy, gx)
fig, ax = plt.subplots(1, 3, figsize=(12, 3.5))
tampil(ax[0], phantom, 'Transisi miring (sintetis)')
tampil(ax[1], mag, 'Magnitudo gradien', vmin=0, vmax=.2, cmap='magma')
tampil(ax[2], np.rad2deg(ang), 'Arah normal (derajat)', vmin=-180, vmax=180, cmap='twilight')
step=13
sel=(mag[::step, ::step]>.045)
qsx=xx[::step, ::step][sel]; qsy=yy[::step, ::step][sel]
ax[0].quiver(qsx, qsy, gx[::step,::step][sel], gy[::step,::step][sel],
             color='yellow', scale=1.0, scale_units='xy', width=.006)
plt.tight_layout(); plt.show()
print('Rata-rata sudut normal pada gradien kuat (derajat):',
      round(float(np.degrees(np.arctan2(np.mean(gy[mag>.05]),np.mean(gx[mag>.05])))),2))

## 5. Kernel Prewitt, Sobel, dan Scharr: derivasi sebagai filter turunan

Turunan pada arah horizontal dapat diperkirakan dengan memadukan diferensiasi antarkolom dan penghalusan antarbaris. Dalam konvensi **korelasi** untuk transisi gelap-ke-terang kiri-ke-kanan, ketiga contoh kernel berikut adalah

$$
K_{\mathrm{Prewitt},x}=\frac{1}{6}
\begin{bmatrix}-1&0&1\\-1&0&1\\-1&0&1\end{bmatrix},\quad
K_{\mathrm{Sobel},x}=\frac{1}{8}
\begin{bmatrix}-1&0&1\\-2&0&2\\-1&0&1\end{bmatrix},
$$

$$
K_{\mathrm{Scharr},x}=\frac{1}{32}
\begin{bmatrix}-3&0&3\\-10&0&10\\-3&0&3\end{bmatrix}.
$$

Normalisasi di atas membuat setiap kernel menghasilkan respons sekitar satu untuk bidang linear $I[m,n]=n$ di interior ($\Delta x=1$ piksel). Kernel untuk arah $y$ diperoleh dengan transpose. *Prewitt* melakukan penghalusan antarbaris seragam, Sobel memberikan bobot tengah lebih tinggi, dan Scharr dirancang agar respons arah lebih isotropik untuk kernel kecil.

**Perhatian implementasi:** definisi normalisasi dan arah dalam pustaka bisa berbeda. `skimage.filters.sobel(image)` menghitung magnitudo gabungan dengan normalisasi sendiri. Oleh karena itu, jangan membandingkan besaran numerik lintas pustaka tanpa memeriksa skala kernel dan definisinya.

In [ ]:
K_pre = np.array([[-1,0,1],[-1,0,1],[-1,0,1]],float)/6
K_sob = np.array([[-1,0,1],[-2,0,2],[-1,0,1]],float)/8
K_sch = np.array([[-3,0,3],[-10,0,10],[-3,0,3]],float)/32
operators = {'Prewitt':K_pre, 'Sobel':K_sob, 'Scharr':K_sch}
plane = np.broadcast_to(np.arange(80,dtype=float), (64,80))
for name,K in operators.items():
    assert np.isclose(K.sum(),0,atol=1e-15)
    assert np.isclose(ndimage.correlate(plane,K,mode='reflect')[8:-8,8:-8].mean(),1,atol=1e-12)
fig,axs=plt.subplots(2,3,figsize=(11.7,6))
for j,(name,K) in enumerate(operators.items()):
    gy=ndimage.correlate(camera,K.T,mode='reflect')
    gx=ndimage.correlate(camera,K,mode='reflect')
    response=np.hypot(gx,gy)
    tampil(axs[0,j],response,f'{name}: magnitude',vmin=0,vmax=.23,cmap='magma')
    axs[1,j].plot(np.arange(65,155),response[110,65:155]);axs[1,j].set(
        xlabel='Kolom',ylabel='Magnitudo (kode/piksel)',title='Profil lokal')
    axs[1,j].grid(alpha=.2)
plt.tight_layout();plt.show()

## 6. Rotasi, anisotropi, dan pengujian ketergantungan arah

Sebuah filter turunan ideal di ruang kontinu mengukur gradien dengan besar yang sama untuk transisi identik yang diputar. Kernel diskret kecil tidak sepenuhnya isotropik: hasilnya bergantung pada orientasi garis terhadap grid, ketebalan blur, dan interpolasi.

Untuk membandingkan operator secara adil, kita membuat transisi sintetis $I(x,y)=a+b\Phi(d/\sigma)$, di mana $d=x\cos\phi+y\sin\phi$ adalah jarak bertanda dari batas lurus, dan $\Phi$ CDF Gaussian. Magnitudo teoretis pada pusat tepi adalah $|b|/(\sqrt{2\pi}\sigma)$ bila PSF Gaussian dan sampling titik menjadi pendekatan yang layak.

Eksperimen ini menilai variasi respons puncak terhadap $\phi$. Ini **bukan bukti** bahwa sebuah operator selalu lebih baik di seluruh citra, resolusi, dan model noise.

In [ ]:
N=120; Y,X=np.mgrid[:N,:N]; angles=np.arange(0,180,15)
rows=[]
for deg in angles:
    phi=np.deg2rad(deg)
    dist=(X-(N-1)/2)*np.cos(phi)+(Y-(N-1)/2)*np.sin(phi)
    sim=.18+.64*special.ndtr(dist/1.35)
    for name,K in operators.items():
        gx=ndimage.correlate(sim,K,mode='reflect')
        gy=ndimage.correlate(sim,K.T,mode='reflect')
        # maksimum di wilayah tengah, jauh dari batas gambar
        peak=np.quantile(np.hypot(gx,gy)[30:-30,30:-30], .99)
        rows.append({'Operator':name,'Sudut (derajat)':deg,'Respons P99':peak})
rot=pd.DataFrame(rows)
fig,ax=plt.subplots(figsize=(9.5,3.5))
for name,z in rot.groupby('Operator'):
    ax.plot(z['Sudut (derajat)'],z['Respons P99']/z['Respons P99'].mean(),'-o',label=name)
ax.set(xlabel='Sudut normal terhadap sumbu kolom (derajat)',
       ylabel='Respons P99 / rata-rata tiap operator',title='Sensitivitas orientasi (sintetis)')
ax.legend();ax.grid(alpha=.2);plt.tight_layout();plt.show()
print(rot.groupby('Operator')['Respons P99'].agg(['mean','std']).round(4).to_string())

## 7. Diferensiasi dan Gaussian scale-space

Pada citra noisy, operator diferensial memperbesar variasi frekuensi tinggi. Salah satu pendekatan terpenting adalah menghitung gradien setelah penghalusan Gaussian:

$$
\frac{\partial}{\partial x}(G_\sigma*I)
=\left(\frac{\partial G_\sigma}{\partial x}\right)*I.
$$

Ini sah jika operasi dapat dipertukarkan di bawah kondisi regularitas yang sesuai. Kernel turunan Gaussian satu dimensi ialah

$$
\frac{dG_\sigma(x)}{dx}=-\frac{x}{\sigma^2}G_\sigma(x),\quad
G_\sigma(x)=\frac{1}{\sqrt{2\pi}\sigma}
\exp\!\left(-\frac{x^2}{2\sigma^2}\right).
$$

Parameter $\sigma$ mengatur **skala deteksi**. Skala kecil lebih peka pada detail tetapi rentan noise; skala besar meningkatkan penekanan noise namun dapat menyatukan tepi yang berdekatan, menumpulkan sudut, dan memperburuk lokalisasi bentuk tertentu. Jika PSF dan penghalusan tambahan keduanya Gaussian serta independen dalam model konvolusional, perkiraan lebar efektif memenuhi $\sigma_{\mathrm{ef}}^2\approx\sigma_{\mathrm{PSF}}^2+\sigma_{\mathrm{filter}}^2$.

In [ ]:
# Skala gradien berubah, sementara data pengukuran tetap sama
rr = np.random.default_rng(SEED+5)
patch = camera[:180,:180]
noisy = np.clip(ndimage.gaussian_filter(patch,1.15)+rr.normal(0,.035,patch.shape),0,1)
scales=[0.0,.65,1.3,2.5]
fig,axs=plt.subplots(2,len(scales),figsize=(12,6.0))
for j,s in enumerate(scales):
    sm=noisy if s==0 else ndimage.gaussian_filter(noisy,s,mode='reflect')
    gy,gx=np.gradient(sm)
    mag=np.hypot(gx,gy)
    tampil(axs[0,j],sm,f'Input, σ={s}',vmin=0,vmax=1)
    tampil(axs[1,j],mag,f'Gradien, σ={s}',vmin=0,vmax=.16,cmap='magma')
plt.tight_layout();plt.show()

## 8. ESF, LSF, PSF dan MTF dari citra tepi

Sebuah pengukuran tepi dapat digunakan untuk mengkarakterisasi ketajaman sistem di bawah asumsi tertentu. Untuk batas ideal satu dimensi gelap-ke-terang, citra terukur membentuk *edge spread function* (ESF). Turunan pertama terhadap arah normal tepi menghasilkan *line spread function* (LSF). Dalam sistem linear, invarian pergeseran, dan tepi ideal yang cukup panjang, LSF adalah integral PSF 2D sepanjang arah sejajar tepi:

$$
\mathrm{LSF}(x)\propto\int_{-\infty}^{\infty}h(x,y)\,dy,
\qquad
\mathrm{MTF}(\nu)=\frac{|\mathcal F\{\mathrm{LSF}\}(\nu)|}{|\mathcal F\{\mathrm{LSF}\}(0)|}.
$$

Dalam contoh Gaussian satu dimensi, PSF marginal memiliki standar deviasi $\sigma$, sehingga

$$
\mathrm{MTF}(\nu)=\exp(-2\pi^2\sigma^2\nu^2),
$$

dengan $\nu$ dalam siklus/piksel. Praktiknya memerlukan perhatian pada *pixel aperture*, slanted-edge oversampling, koreksi noise, sampling, penempatan ROI, dan *windowing*. ESF dari suatu tepi yang tidak ideal tidak otomatis merupakan karakterisasi PSF kamera yang dapat ditelusuri ke standar metrologi.

In [ ]:
# Kurva ESF -> LSF -> MTF sintetis dengan PSF Gaussian
z=np.linspace(-24,24,481); dx=z[1]-z[0]
edge_sigma=2.0; esf=special.ndtr(z/edge_sigma)
lsf=np.gradient(esf,dx)
lsf=lsf/trapezoid(lsf,z)
freq=np.fft.rfftfreq(len(lsf),d=dx)
mtf=np.abs(np.fft.rfft(np.fft.ifftshift(lsf)))
mtf=mtf/mtf[0]
theory=np.exp(-2*np.pi**2*edge_sigma**2*freq**2)
fig,axs=plt.subplots(1,3,figsize=(12,3.3))
axs[0].plot(z,esf);axs[0].set(xlabel='Posisi (piksel)',ylabel='ESF',title='Edge spread function')
axs[1].plot(z,lsf);axs[1].set(xlabel='Posisi (piksel)',ylabel='LSF (piksel⁻¹)',title='Line spread function')
axs[2].plot(freq,mtf,label='Dari LSF');axs[2].plot(freq,theory,'--',label='Model Gaussian')
axs[2].set(xlim=(0,.5),ylim=(0,1.05),xlabel='Frekuensi (siklus/piksel)',ylabel='MTF',title='Perbandingan MTF')
axs[2].legend()
for ax in axs:ax.grid(alpha=.2)
plt.tight_layout();plt.show()
mask_mid=(freq>0)&(freq<.4)
print('Galat maksimum MTF vs model Gaussian (0 < ν < 0.4):',
      f'{np.max(np.abs(mtf[mask_mid]-theory[mask_mid])):.5f}')

## 9. Mengapa threshold tunggal tidak cukup?

Setelah menghitung magnitudo gradien $G[m,n]$, aturan sederhana $G>\tau$ memberi peta kandidat tepi. Namun satu transisi yang melebar dapat menghasilkan **beberapa piksel tebal**. Noise juga dapat memunculkan kandidat terisolasi. Mengubah ambang menghasilkan kompromi deteksi: ambang rendah meningkatkan sensitivitas tetapi memperbanyak *false positive*, sedangkan ambang tinggi dapat menghilangkan kontur lemah.

Dua perangkat konseptual penting adalah:

- **Non-maximum suppression (NMS):** di sepanjang arah normal gradien, hanya puncak lokal magnitudo yang dipertahankan agar respons menjadi lebih tipis. Implementasi akurat harus menangani arah non-integer melalui interpolasi atau pendekatan yang jelas.
- **Hysteresis:** kandidat dengan respons kuat menjadi penanda awal; kandidat lebih lemah dipertahankan jika tersambung secara spasial dengan kandidat kuat menurut aturan ketetanggaan. Hasilnya bergantung pada dua ambang, definisi keterhubungan, dan preprocessing.

Sebuah kontur yang tipis secara digital belum tentu memperlihatkan ketidakpastian lokasi yang kecil secara fisik. *Single-pixel edge* merepresentasikan keputusan algoritmik, bukan titik geometris eksak.

In [ ]:
# Demonstrasi NMS 1D sederhana dan hysteresis pada deret sintetis
r=np.linspace(0,140,141)
strength=(.9*np.exp(-.5*((r-29)/4)**2)+
          .38*np.exp(-.5*((r-73)/5)**2)+
          .16*np.exp(-.5*((r-102)/3)**2))
strength+=.04*np.random.default_rng(SEED+7).random(r.size)
peaks=np.zeros_like(strength,dtype=bool)
peaks[1:-1]=(strength[1:-1]>strength[:-2])&(strength[1:-1]>=strength[2:])
lo,hi=.13,.32
strong=peaks&(strength>=hi); weak=peaks&(strength>=lo)
fig,ax=plt.subplots(figsize=(10,3.2))
ax.plot(r,strength,color='0.3',label='Magnitudo gradien (ilustrasi)')
ax.scatter(r[weak],strength[weak],s=25,facecolors='none',edgecolors='orange',label='Puncak ≥ ambang rendah')
ax.scatter(r[strong],strength[strong],s=35,c='crimson',label='Puncak ≥ ambang tinggi')
ax.axhline(lo,ls=':',color='orange');ax.axhline(hi,ls='--',color='crimson')
ax.set(xlabel='Posisi',ylabel='Respons',title='Penyaringan puncak dan dua ambang (ilustrasi 1D)')
ax.legend(fontsize=8);ax.grid(alpha=.2);plt.tight_layout();plt.show()
print('Jumlah puncak pada dua ambang:',int(weak.sum()),int(strong.sum()))
print('Catatan: diagram 1D ini belum menerapkan hysteresis keterhubungan 2D.')

## 10. Algoritma Canny: deteksi, lokalisasi, dan respons tunggal

*Canny edge detector* dirancang berdasarkan pertimbangan deteksi yang baik, lokalisasi yang baik, dan respons yang tidak berlipat pada sebuah tepi. Implementasi yang umum memiliki alur:

1. Penghalusan Gaussian dengan parameter $\sigma$.
2. Perhitungan gradien (umumnya turunan diskret berbasis operator seperti Sobel).
3. NMS pada magnitudo gradien menurut arah normal.
4. Pemisahan kandidat lemah dan kuat dengan dua ambang.
5. Hysteresis yang menghubungkan kandidat lemah dengan kontur kuat.

Pada `skimage.feature.canny`, `use_quantiles=True` menafsirkan ambang sebagai **kuantil magnitudo**, bukan nilai intensitas absolut. Ambang relatif semacam itu tidak otomatis setara antarperangkat atau domain citra.

Keluaran Canny bersifat **biner**. Posisi integer tidak menyatakan batas subpiksel; untuk estimasi geometris presisi, dibutuhkan tahap lokalisasi atau pemasangan model tambahan. Pada citra dengan batas saling berdekatan, *threshold hysteresis* dan Gaussian smoothing dapat menyambungkan atau memisahkan kontur secara tidak sesuai konteks fisis.

In [ ]:
# Bandingkan Sobel magnitudo dan Canny pada citra contoh, termasuk variasi sigma
sample=camera
sob=filters.sobel(sample)
canny_sigma=[.8,1.5,2.8]
fig,axs=plt.subplots(1,5,figsize=(14,3.3))
tampil(axs[0],sample,'Input (camera)')
tampil(axs[1],sob,'Sobel |gradien|',vmin=0,vmax=.22,cmap='magma')
for j,s in enumerate(canny_sigma,start=2):
    edges=feature.canny(sample,sigma=s,low_threshold=.1,high_threshold=.2,mode='reflect')
    tampil(axs[j],edges,f'Canny σ={s}',vmin=0,vmax=1)
    print(f'Canny sigma={s}: {edges.sum()} piksel tepi; pecahan {edges.mean():.4f}')
plt.tight_layout();plt.show()

## 11. Turunan kedua, Laplacian, dan zero crossing

Gradien pertama mengidentifikasi perubahan intensitas. Turunan kedua menyorot perubahan **laju** gradien. Laplacian dua dimensi adalah

$$
\nabla^2 I=\frac{\partial^2 I}{\partial x^2}+\frac{\partial^2 I}{\partial y^2}.
$$

Untuk transisi ideal yang diblur simetris, lokasi tepi dapat dihubungkan dengan *zero crossing* dari turunan kedua. Akan tetapi, turunan kedua sangat sensitif terhadap noise. Pendekatan *Laplacian of Gaussian* (LoG) menggabungkan penghalusan dan diferensiasi:

$$
\nabla^2(G_\sigma*I)=(\nabla^2G_\sigma)*I.
$$

**Catatan penting:** tidak setiap zero crossing merupakan tepi objek. Di dekat ekstrem tekstur, noise, atau batas ROI dapat muncul banyak perubahan tanda. Magnitudo respons dan keterhubungan spasial tetap perlu dinilai; dalam beberapa aplikasi, tepi kedua dan pertama memberi informasi komplementer.

In [ ]:
# Zero-crossing LoG dibatasi oleh ambang amplitudo lokal agar lebih bermakna
img_small = camera[40:200, 40:200]
noisy_small = np.clip(img_small + rng.normal(0,.035,img_small.shape),0,1)
log_scales=[0.7,1.4,2.7]
fig,axs=plt.subplots(2,3,figsize=(11,6))
for j,s in enumerate(log_scales):
    lap=ndimage.gaussian_laplace(noisy_small,sigma=s,mode='reflect')
    crosses=np.zeros_like(lap,dtype=bool)
    crosses[:,1:]=(lap[:,1:]*lap[:,:-1] < 0)
    crosses[1:,:]|=(lap[1:,:]*lap[:-1,:]<0)
    # batas amplitudo adalah pilihan ilustratif, bukan aturan universal
    strong=crosses & (ndimage.maximum_filter(abs(lap),size=3) > np.quantile(abs(lap),.75))
    tampil(axs[0,j],lap,f'LoG σ={s}',vmin=-.07,vmax=.07,cmap='coolwarm')
    tampil(axs[1,j],strong,f'Zero crossing + seleksi, σ={s}')
plt.tight_layout();plt.show()

## 12. Pengaruh batas citra, ROI, dan artefak numerik

Operator diferensial membutuhkan sampel tetangga. Pada tepi bidang pandang, interpolasi di luar domain melalui `reflect`, `nearest`, atau `constant` bukan informasi nyata tentang objek. Padding konstan yang berbeda dari intensitas latar dapat menciptakan tepi buatan; `reflect` dapat menyembunyikan diskontinuitas yang sesungguhnya tepat pada batas sensor.

Karena itu, protokol metrologi sebaiknya menyatakan: batas ROI, *mask* valid, strategi padding, dan apakah piksel dekat batas dikeluarkan. Data yang *clipped* atau tersaturasi juga mengubah gradien secara nonlinear. Pada citra RGB yang telah dikenai gamma, turunan kode warna tidak identik dengan turunan radiansi atau luminansi linear.

In [ ]:
# Padding constant vs reflect dapat menciptakan tepi semu
F=np.ones((86,110),dtype=float)*.77
F[15:68,28:80]=.27
modes=['constant','reflect','nearest']
fig,ax=plt.subplots(1,3,figsize=(10.5,3.1))
for a,mode in zip(ax,modes):
    gx=ndimage.sobel(F,axis=1,mode=mode,cval=0)/8
    gy=ndimage.sobel(F,axis=0,mode=mode,cval=0)/8
    tampil(a,np.hypot(gx,gy),f'Padding: {mode}',vmin=0,vmax=.4,cmap='magma')
plt.tight_layout();plt.show()
print('Gradient maksimum batas kiri (constant):',
      round(float(np.hypot(ndimage.sobel(F,axis=1,mode='constant',cval=0)/8,
                            ndimage.sobel(F,axis=0,mode='constant',cval=0)/8)[:,0].max()),3))

## 13. Lokalisasi subpiksel dari puncak gradien

Banyak algoritma mengembalikan koordinat tepi berupa bilangan bulat. Untuk pengukuran geometri, posisi dapat diestimasi kontinu menggunakan pencocokan kurva di sekitar puncak, misalnya pendekatan parabola pada tiga sampel respons $r_{-1},r_0,r_{+1}$. Puncak parabola berada pada perpindahan lokal

$$
\delta=\frac{r_{-1}-r_{+1}}{2(r_{-1}-2r_0+r_{+1})},
\qquad \widehat x=x_0+\delta,
$$

bila kelengkungan negatif, puncak cukup terisolasi, dan interpolasi lokal dapat diterima. Membatasi $\delta$ pada $[-0.5,0.5]$ mencegah ekstrapolasi besar ketika model parabola degenerat.

Interpolasi dapat mengurangi efek kuantisasi posisi pada kondisi baik, tetapi **tidak menciptakan informasi fisik di luar data**. Bias tetap muncul akibat PSF asimetris, *rolling shutter*, perubahan iluminasi, saturasi, *correlated noise*, atau model kurva yang tidak sesuai. Perbedaan *precision* (sebaran pengulangan) dan *accuracy* (kedekatan ke posisi benar) menjadi penting.

In [ ]:
# Bandingkan posisi integer dan interpolasi subpiksel pada langkah Gaussian
subrng=np.random.default_rng(SEED+13)
xb=67.37; sigma_ideal=1.5
xx1=np.arange(135)
ideal=.18+.57*special.ndtr((xx1-xb)/sigma_ideal)

def interpolasi_parabola(profil, idx):
    if idx<1 or idx>=len(profil)-1: return float(idx)
    a,b,c=float(profil[idx-1]),float(profil[idx]),float(profil[idx+1])
    curv=a-2*b+c
    if curv>=-1e-12:return float(idx)
    delta=np.clip(.5*(a-c)/curv,-.5,.5)
    return float(idx+delta)

records=[];first=None
for j in range(350):
    noisy1=ideal+subrng.normal(0,.018,size=ideal.shape)
    smooth1=ndimage.gaussian_filter1d(noisy1,1.0,mode='reflect')
    response=np.gradient(smooth1)
    idx=45+np.argmax(response[45:90])
    x_sub=interpolasi_parabola(response,idx)
    records.append((idx, x_sub))
    if j==0:first=(noisy1,response,idx,x_sub)
arr=np.array(records)
fig,axs=plt.subplots(1,2,figsize=(11.5,3.6))
axs[0].plot(xx1,first[0],label='Citra terukur 1D')
axs[0].axvline(xb,color='k',ls='--',label='Batas fisik model')
axs[0].set(xlim=(57,78),xlabel='Posisi (piksel)',ylabel='Intensitas');axs[0].legend()
axs[1].hist(arr[:,0]-xb,bins=22,alpha=.6,label='Piksel integer')
axs[1].hist(arr[:,1]-xb,bins=22,alpha=.6,label='Interpolasi subpiksel')
axs[1].set(xlabel='Galat posisi (piksel)',ylabel='Frekuensi',title='350 pengulangan independen')
axs[1].legend();plt.tight_layout();plt.show()
for name,v in [('Integer',arr[:,0]),('Subpiksel',arr[:,1])]:
    print(f'{name}: bias={np.mean(v-xb):+.3f} piksel, simpangan baku={np.std(v,ddof=1):.3f}, MAE={np.mean(abs(v-xb)):.3f}')

## 14. Pengaruh skala dan ketidakpastian: ulangan bukan jaminan akurasi

Untuk suatu batas fisik dengan posisi benar $x_0$, sebuah estimator mempunyai galat $e=\widehat x-x_0$. Kuadrat galat rata-rata memenuhi identitas

$$
\operatorname{MSE}(\widehat x)=\underbrace{\big(\mathbb E[\widehat x]-x_0\big)^2}_{\mathrm{bias}^2}
+\underbrace{\operatorname{Var}(\widehat x)}_{\text{presisi statistik}},
$$

dengan ekspektasi terhadap sumber acak yang dinyatakan. Penghalusan dapat menurunkan varians estimator namun mengubah bias. Penggunaan banyak baris citra **tidak** menjamin ketidakpastian turun mengikuti $1/\sqrt N$ karena pencahayaan, PSF, dan noise antarbari​s dapat berkorelasi.

Bandingkan parameter yang telah dipilih sebelumnya pada pengulangan independen, bukan memilih parameter terbaik setelah melihat data uji. Ketika model simulasi diketahui, kita dapat mengukur bias terhadap *ground truth*. Pada data lapangan, posisi benar perlu berasal dari standar referensi atau metode pembanding yang ketertelusurannya diketahui.

In [ ]:
# Trade-off skala gradien dan kesalahan posisi dalam simulasi 1D
B=150; simrng=np.random.default_rng(SEED+14)
rad=np.arange(130); truth_x=65.2
sigmas=[.0,.45,.9,1.6,2.6]
rows=[]
for s in sigmas:
    est=[]
    for _ in range(B):
        line=.28+.39*special.ndtr((rad-truth_x)/1.7)+simrng.normal(0,.035,rad.size)
        z=line if s==0 else ndimage.gaussian_filter1d(line,s,mode='reflect')
        gr=np.gradient(z)
        k=45+np.argmax(gr[45:87])
        est.append(interpolasi_parabola(gr,k))
    est=np.asarray(est)
    rows.append({'sigma':s,'bias_px':np.mean(est-truth_x),'sd_px':est.std(ddof=1),
                 'mae_px':np.mean(abs(est-truth_x))})
scales_df=pd.DataFrame(rows)
display(scales_df.round(4))
fig,ax=plt.subplots(figsize=(8.8,3.2))
ax.plot(scales_df.sigma,scales_df.mae_px,'-o',label='MAE posisi')
ax.plot(scales_df.sigma,abs(scales_df.bias_px),'-s',label='|bias|')
ax.plot(scales_df.sigma,scales_df.sd_px,'-^',label='simpangan baku')
ax.set(xlabel='Sigma penghalusan tambahan (piksel)',ylabel='Galat posisi (piksel)',
       title='Skala optimal bergantung tujuan dan model noise')
ax.legend();ax.grid(alpha=.2);plt.tight_layout();plt.show()

## 15. Hubungan tepi dengan segmentasi dan morfologi: jembatan menuju materi 05

Deteksi tepi mengidentifikasi kandidat transisi intensitas; **segmentasi** mencoba menentukan wilayah atau objek sesuai definisi kelas/instans. Keduanya tidak ekuivalen: kontur tidak tertutup, noise, refleksi, dan kontak antarobjek dapat membuat penghitungan komponen terhubung berbeda dari jumlah benda nyata.

*Erosion* dan *dilation* pada mask biner didefinisikan oleh elemen struktur $B$. *Opening* (erosi lalu dilasi) cenderung menghapus komponen kecil dan tonjolan; *closing* (dilasi lalu erosi) dapat menutup celah. Efek konkret bergantung pada ukuran, geometri $B$, konektivitas, dan konvensi batas; tidak ada jaminan mempertahankan area fisik maupun jumlah objek.

Bagian ini melestarikan kompetensi morfologi dari notebook 04 versi awal sebagai **pengantar singkat**. Pengukuran wilayah, anotasi, dan metrik tingkat objek akan diperdalam pada materi 05 dan materi pengukuran ilmiah lanjutan.

In [ ]:
coins=img_as_float(data.coins())
threshold=filters.threshold_otsu(coins)
mask=coins>threshold
opened=morphology.opening(mask,morphology.disk(2))
closed=morphology.closing(opened,morphology.disk(3))
coin_edge=feature.canny(coins,sigma=1.7)
fig,axs=plt.subplots(1,4,figsize=(13,3.7))
for ax,z,title in zip(axs,[coin_edge,mask,opened,closed],
                      ['Canny: kandidat kontur','Mask Otsu','Opening r=2','Closing setelah opening r=3']):
    tampil(ax,z,title)
plt.tight_layout();plt.show()
for name,z in [('Otsu',mask),('Opening',opened),('Closing',closed)]:
    n=measure.label(z,connectivity=2).max()
    print(f'{name}: {n} komponen terhubung; bukan otomatis {n} koin fisik')

## 16. Metrik evaluasi untuk tepi dan posisi geometris

**Metrik klasifikasi piksel biasa tidak selalu cocok untuk batas geometris yang sangat tipis.** Dua kontur benar yang bergeser satu piksel dapat memberikan *pixel-wise F1* rendah walaupun secara metrologi mungkin masih dalam toleransi. Untuk posisi batas $x(y)$ dengan *ground truth* yang diketahui, ukuran kesalahan geometris adalah

$$
\mathrm{MAE}_x=\frac{1}{N}\sum_{y=1}^N|\widehat x(y)-x_0(y)|,
\qquad \mathrm{RMSE}_x=\sqrt{\frac{1}{N}\sum_{y=1}^N(\widehat x(y)-x_0(y))^2}.
$$

Jika estimator boleh gagal menghasilkan titik untuk sebagian baris, **coverage** harus turut dilaporkan. MAE pada baris yang berhasil saja dapat mengecil secara menyesatkan saat metode sering gagal. Karena itu, kita juga menggunakan *penalized MAE*: baris tanpa titik tepi diberi penalti jarak tertentu yang dipilih **sebelum** evaluasi.

Untuk perbandingan mask tepi umum, *tolerance-based boundary precision/recall* dapat mendefinisikan kecocokan dalam radius $r$. Namun pencocokan tanpa korespondensi satu-ke-satu dapat memberi hitungan ganda; saat kepentingan objek tinggi, gunakan pencocokan kontur/objek yang eksplisit.

In [ ]:
# F1 piksel vs kecocokan tepi berbasis toleransi
mask_a=np.zeros((75,120),bool);mask_b=np.zeros_like(mask_a)
mask_a[12:65,52]=True
mask_b[12:65,53]=True
intersection=(mask_a&mask_b).sum()
pixel_f1=2*intersection/(mask_a.sum()+mask_b.sum())
dist_a=ndimage.distance_transform_edt(~mask_a)
dist_b=ndimage.distance_transform_edt(~mask_b)
r=1.0
prec=(dist_a[mask_b]<=r).mean()
rec=(dist_b[mask_a]<=r).mean()
tol_f1=2*prec*rec/(prec+rec) if prec+rec else 0
print(f'Pixel-wise F1={pixel_f1:.3f}; F1 toleransi {r:g} piksel={tol_f1:.3f}')
fig,axs=plt.subplots(1,3,figsize=(10,3))
for a,z,t in zip(axs,[mask_a,mask_b,np.dstack([mask_a.astype(float),mask_b.astype(float),np.zeros_like(mask_a,float)])],
                 ['Kontur referensi','Kontur digeser 1 piksel','Overlay merah vs hijau']):
    if z.ndim==3: a.imshow(z)
    else: tampil(a,z,t)
    a.set_title(t);a.set_xlim(43,61);a.set_ylim(70,5)
plt.tight_layout();plt.show()
assert pixel_f1==0 and tol_f1==1.0

## 17. Studi kasus integratif — Pengukuran posisi batas material pada citra sensor

**Masalah engineering.** Sebuah kamera mengamati batas dua material dengan reflektansi berbeda. Posisi batas sedikit bergelombang; pencahayaan tidak sepenuhnya seragam; kamera mengaburkan transisi; elektron terukur dipengaruhi shot noise dan read noise. Kita ingin memperkirakan posisi batas $x_0(y)$ sepanjang spesimen dan melaporkan kesalahan dalam piksel dan milimeter.

Model sintetis kontinu yang diproyeksikan ke grid adalah

$$
I_0(x,y)=L(x,y)\big[a+b\,\Phi\big((x-x_0(y))/\sigma_{\mathrm{PSF}}\big)\big]+T(x,y),
$$

$$
N_e(x,y)\sim\operatorname{Poisson}\big(\mu_e(x,y)\big),\qquad
C(x,y)=Q_{12}\!\left(\frac{N_e+\epsilon_r-e_{\mathrm{off}}}{g_e}\right),
$$

dengan $L$ medan iluminasi, $T$ komponen tekstur lemah, $\Phi$ CDF normal standar, $\mu_e=e_{\mathrm{off}}+g_e I_0$, $\epsilon_r$ read noise Gaussian, dan $Q_{12}$ kuantisasi 12-bit ke rentang normalisasi `[0,1]` disertai *clipping*. Persamaan adalah model penyederhanaan; *pixel aperture* terserap secara efektif ke dalam $\sigma_{\mathrm{PSF}}$, bukan dimodelkan sebagai integrasi area terpisah.

**Batas klaim:** dataset buatan sendiri, bukan citra logam nyata, bukan hasil kalibrasi kamera, dan bukan pengujian kelayakan sistem industri. Pitch $p=0.08$ mm/piksel di bawah merupakan asumsi demonstrasi.

**Protokol:** 9 spesimen validasi untuk memilih parameter, kemudian 13 spesimen pengujian yang sepenuhnya berbeda dan tidak digunakan untuk memilih konfigurasi. Unit replikasi utama untuk inferensi adalah **spesimen**, bukan setiap baris piksel.

In [ ]:
# Generator data lengkap, transparan, dan deterministik
HEIGHT, WIDTH = 128, 144
PITCH_MM = .08  # asumsi ilustratif, bukan hasil kalibrasi
ROI_LO, ROI_HI = 55, 92  # jendela posisi berdasarkan fixturing nominal
MISS_PENALTY_PX = 10.0

def buat_spesimen(seed):
    rr=np.random.default_rng(seed)
    y,x=np.mgrid[:HEIGHT,:WIDTH]
    phi=rr.uniform(0,2*np.pi)
    center=72+rr.uniform(-1.8,1.8)
    edge=center+2.7*np.sin(2*np.pi*y[:,0]/70+phi)+1.1*np.sin(2*np.pi*y[:,0]/39+.4*phi)
    psf_sigma=rr.uniform(.9,2.4)
    contrast=rr.uniform(.23,.37)
    illumination=(.86+.08*(x/WIDTH)+.09*np.sin(y/HEIGHT*3.2+phi))
    clean=illumination*(.24+contrast*special.ndtr((x-edge[:,None])/psf_sigma))
    clean+=.009*np.sin(x/7.8+y/19+.2*phi)
    clean=np.clip(clean,0,1)
    electrons_offset=80.0; gain=950.0
    mu=electrons_offset+gain*clean
    measured_e=rr.poisson(mu)+rr.normal(0,12.0,(HEIGHT,WIDTH))
    observed=np.clip((measured_e-electrons_offset)/gain,0,1)
    image=np.round(observed*4095)/4095
    return {'image':image,'edge':edge,'clean':clean,'psf_sigma':psf_sigma,
            'contrast':contrast,'seed':seed}

validation=[buat_spesimen(SEED+1000+j) for j in range(9)]
test=[buat_spesimen(SEED+5000+j) for j in range(13)]
assert {d['seed'] for d in validation}.isdisjoint({d['seed'] for d in test})
case=validation[0]
fig,axs=plt.subplots(1,3,figsize=(12,3.4))
tampil(axs[0],case['clean'],'Intensitas laten (sintetis)')
tampil(axs[1],case['image'],'Sensor + noise + ADC (sintetis)')
axs[1].plot(case['edge'],np.arange(HEIGHT),color='yellow',lw=1.1,label='Batas model')
axs[1].legend(fontsize=7,loc='lower left')
axs[2].plot(np.arange(WIDTH),case['image'][60],label='Terukur')
axs[2].plot(np.arange(WIDTH),case['clean'][60],label='Sebelum noise')
axs[2].axvline(case['edge'][60],ls='--',color='k',label='Batas model')
axs[2].set(xlim=(48,101),xlabel='Kolom (piksel)',ylabel='Intensitas',title='Profil baris y=60')
axs[2].legend(fontsize=7);axs[2].grid(alpha=.2)
plt.tight_layout();plt.show()
print(f'Jumlah spesimen: validasi={len(validation)}, uji={len(test)}; ROI x=[{ROI_LO},{ROI_HI})')

### 17A. Definisikan estimator yang tidak membaca *ground truth*

Pada sistem ini arah transisi **diketahui** (kiri gelap, kanan lebih terang), dan fixturing mekanis menempatkan batas dalam ROI tetap yang diketahui dari desain proses. Dengan prior tersebut, estimator mencari puncak **gradien horizontal bertanda positif** pada setiap baris. Pendekatan ini bukan algoritma segmentasi umum, dan tidak boleh diam-diam digunakan pada citra yang orientasi tepinya tidak diketahui.

Kita bandingkan:

- **Beda tengah:** turunan horizontal sederhana tanpa penghalusan tambahan.
- **Sobel dan Scharr:** operator spasial yang boleh diberi Gaussian smoothing.
- **Derivative-of-Gaussian (DoG):** turunan citra yang dihaluskan dengan Gaussian.
- **Canny (integer):** memilih kandidat kontur dari masker biner, lalu menentukan kandidat di dalam ROI dengan respons gradien terbesar. Baris tanpa kandidat dinyatakan *missing*.
- **Canny + subpiksel:** memakai kandidat Canny yang sama, lalu menginterpolasi puncak gradien untuk menguji sumbangan tahap lokalisasi tambahan.

Metode gradien menginterpolasi puncak dengan parabola; Canny ditampilkan baik sebagai output integer maupun sebagai *pipeline* yang diberi tahap subpiksel. Perbandingan tersebut **menguji pipeline**, bukan membuktikan bahwa satu operator selalu unggul, karena tahap pascaproses dapat berbeda.

In [ ]:
# Implementasi estimator. Tidak ada parameter/akses ke scene['edge'] pada fungsi ini.
def estimasi_batas(image, metode, sigma=0.0, low=.55, high=.85):
    sm=ndimage.gaussian_filter(image,sigma=sigma,mode='reflect') if sigma>0 else image
    if metode=='Beda tengah':
        gx=np.gradient(sm,axis=1)
    elif metode=='Sobel':
        gx=ndimage.correlate(sm,K_sob,mode='reflect')
    elif metode=='Scharr':
        gx=ndimage.correlate(sm,K_sch,mode='reflect')
    elif metode=='DoG':
        gx=ndimage.gaussian_filter(image,sigma=max(sigma,.35),order=(0,1),mode='reflect')
    elif metode in ('Canny','Canny + subpiksel'):
        gx=ndimage.correlate(sm,K_sob,mode='reflect')
    else: raise ValueError(f'Metode tidak dikenal: {metode}')
    response=np.maximum(gx,0.0)
    if metode in ('Canny','Canny + subpiksel'):
        edge_mask=feature.canny(image,sigma=max(sigma,.45),
                                low_threshold=low,high_threshold=high,
                                use_quantiles=True,mode='reflect')
        # Respon hanya dinilai untuk posisi yang Canny nyatakan sebagai tepi.
        scores=np.where(edge_mask,response,-np.inf)
        win=scores[:,ROI_LO:ROI_HI]
        idx=np.argmax(win,axis=1)+ROI_LO
        has=np.isfinite(np.max(win,axis=1))
        xhat=(idx.astype(float) if metode=='Canny' else
              np.array([interpolasi_parabola(response[y],int(j)) for y,j in enumerate(idx)]))
        xhat[~has]=np.nan
    else:
        win=response[:,ROI_LO:ROI_HI]
        idx=np.argmax(win,axis=1)+ROI_LO
        xhat=np.array([interpolasi_parabola(response[y],int(j))
                       for y,j in enumerate(idx)],dtype=float)
    return xhat,response

def skor_batas(scene, posisi):
    correct=scene['edge']; valid=np.isfinite(posisi)
    if valid.any():
        mae_valid=np.mean(abs(posisi[valid]-correct[valid]))
    else: mae_valid=np.nan
    penalized=np.where(valid,abs(np.nan_to_num(posisi,nan=0)-correct),MISS_PENALTY_PX)
    return {'coverage':valid.mean(), 'mae_valid_px':mae_valid,
            'mae_penalized_px':penalized.mean(),
            'within_1_5px':np.mean(valid & (abs(np.nan_to_num(posisi,nan=-1000)-correct)<=1.5))}

for metode in ['Beda tengah','Sobel','Scharr','DoG','Canny','Canny + subpiksel']:
    pos,_=estimasi_batas(case['image'],metode,sigma=1.3 if metode!='Beda tengah' else 0)
    print(metode, {k:round(float(v),3) for k,v in skor_batas(case,pos).items()})

### 17B. Seleksi hyperparameter **hanya dari validation set**

Tentukan kandidat sebelum evaluasi, gunakan seluruh spesimen validasi, lalu bekukan satu konfigurasi untuk setiap metode. Karena kriteria utama adalah **kesalahan posisi**, seleksi memakai rata-rata *penalized MAE* per spesimen. Penalti 10 piksel untuk baris yang gagal ditetapkan sejak awal; nilainya bukan estimasi jarak fisik sebenarnya, melainkan biaya keputusan yang harus diuji sensitivitasnya.

**Mengapa tidak memakai PSNR?** Kita tidak sedang merekonstruksi seluruh citra; targetnya adalah posisi geometri. Citra yang lebih mirip secara global tidak menjamin kesalahan tepi lebih kecil. Menggunakan data uji untuk memilih $\sigma$ atau ambang akan menyebabkan *optimistic bias* pada estimasi performa.

In [ ]:
# Kandidat ditentukan sebelum melihat hasil set uji
candidates={
    'Beda tengah':[{'sigma':0.0}],
    'Sobel':[{'sigma':float(s)} for s in [0,.65,1.15,1.85,2.7]],
    'Scharr':[{'sigma':float(s)} for s in [0,.65,1.15,1.85,2.7]],
    'DoG':[{'sigma':float(s)} for s in [.55,.85,1.25,1.85,2.5]],
    'Canny':[{'sigma':float(s),'low':float(lo),'high':float(hi)}
             for s in [.8,1.2,1.8,2.5] for lo,hi in [(.48,.78),(.65,.88)]],
    'Canny + subpiksel':[{'sigma':float(s),'low':float(lo),'high':float(hi)}
             for s in [.8,1.2,1.8,2.5] for lo,hi in [(.48,.78),(.65,.88)]]
}
trials=[];selected={}
for metode,configs in candidates.items():
    for p in configs:
        per_scene=[]
        for s in validation:
            pred,_=estimasi_batas(s['image'],metode,**p)
            per_scene.append(skor_batas(s,pred)['mae_penalized_px'])
        trials.append({'metode':metode,'params':str(p),'val_penalized_px':np.mean(per_scene)})
    group=[r for r in trials if r['metode']==metode]
    best=min(group,key=lambda q:q['val_penalized_px'])
    selected[metode]=next(p for p in configs if str(p)==best['params'])
print('Hasil seleksi validasi (belum menggunakan test):')
for method,params in selected.items():
    q=next(r for r in trials if r['metode']==method and r['params']==str(params))
    print(f'{method:13} params={str(params):43} val MAE penalized={q["val_penalized_px"]:.3f} px')

### 17C. Evaluasi data uji terpisah dan unit statistik

Konfigurasi validasi kini **dibekukan**. Setiap spesimen uji menghasilkan satu nilai kesalahan, satu nilai *coverage*, dan satu probabilitas keberhasilan baris dalam toleransi $1.5$ piksel. Perbandingan rerata dilakukan pada tingkat 13 spesimen uji; nilai per baris tidak dianggap ulangan independen karena memiliki pencahayaan, bentuk, dan parameter sensor bersama.

Konversi ke mm dilakukan dengan $p=0.08$ mm/piksel pada contoh ini. Nilai mm mencerminkan asumsi skala, bukan ketidakpastian kalibrasi alat yang benar-benar diukur. Di bagian berikutnya, kita tambahkan skenario ketidakpastian pitch secara eksplisit.

In [ ]:
# Evaluasi held-out: satu baris hasil per metode per spesimen
results=[]
for metode,p in selected.items():
    for j,s in enumerate(test):
        xhat,_=estimasi_batas(s['image'],metode,**p)
        res=skor_batas(s,xhat)
        results.append({'Metode':metode,'Spesimen':j,**res,
                        'mae_penalized_mm':res['mae_penalized_px']*PITCH_MM,
                        'mae_valid_mm':res['mae_valid_px']*PITCH_MM})
df=pd.DataFrame(results)
summary=df.groupby('Metode').agg(
    MAE_mm=('mae_penalized_mm','mean'),
    MAE_terdeteksi_mm=('mae_valid_mm','mean'),
    Coverage=('coverage','mean'),
    Fraksi_dalam_toleransi=('within_1_5px','mean')
).sort_values('MAE_mm')
print('Hasil held-out (13 spesimen, bukan 1664 baris independen):')
display(summary.round(4))
assert (df.coverage.between(0,1)).all()
assert (df.within_1_5px.between(0,1)).all()
assert len(df)==len(test)*len(selected)

### 17D. Ketidakpastian antarspesimen melalui bootstrap

Kita gunakan *nonparametric bootstrap* tingkat spesimen untuk meringkas variasi rerata *penalized MAE* pada sampel uji sintetis. Bootstrap ini mengestimasi **ketidakpastian sampling terhadap keluarga simulator yang sama**, bukan validasi lintas jenis kamera, kondisi lapangan, atau material nyata.

Pada perbandingan berpasangan, gunakan **indeks bootstrap spesimen yang sama** untuk semua metode agar variasi perbedaan mempertahankan desain berpasangan. Interval persentil 95% adalah ilustrasi, bukan pernyataan probabilitas Bayesian atau bukti signifikansi universal.

In [ ]:
# Bootstrap berpasangan pada 13 spesimen
methods=list(selected)
mat=np.column_stack([df[df.Metode==m].sort_values('Spesimen').mae_penalized_mm.to_numpy()
                     for m in methods])
brng=np.random.default_rng(SEED+888)
ix=brng.integers(0,len(test),size=(3000,len(test)))
boot=mat[ix].mean(axis=1)
ci=np.quantile(boot,[.025,.975],axis=0)
ci_table=pd.DataFrame({'Metode':methods,'Rerata MAE (mm)':mat.mean(axis=0),
                       'CI 2.5% (mm)':ci[0],'CI 97.5% (mm)':ci[1]})
display(ci_table.sort_values('Rerata MAE (mm)').round(4))
order=np.argsort(mat.mean(axis=0))
if len(order)>=2:
    j1,j2=order[:2]
    delta=boot[:,j1]-boot[:,j2]
    print('Selisih MAE (metode terendah minus kedua-terendah):',methods[j1],'-',methods[j2],
          'mean=',round(float(np.mean(delta)),5),'mm; CI=',np.round(np.quantile(delta,[.025,.975]),5))
fig,ax=plt.subplots(figsize=(9.5,3.4))
for j,m in enumerate(methods):
    ax.errorbar(mat[:,j].mean(),j,
                xerr=np.array([[mat[:,j].mean()-ci[0,j]],[ci[1,j]-mat[:,j].mean()]]),
                fmt='o',capsize=4,label=m)
ax.set(yticks=np.arange(len(methods)),yticklabels=methods,xlabel='Penalized MAE (mm)',
       title='Rerata dan CI bootstrap per spesimen (simulasi)')
ax.grid(alpha=.2,axis='x');plt.tight_layout();plt.show()

### 17E. Diagnostik kegagalan: visualisasi posisi batas dan residual

Tabel performa sering menyembunyikan lokasi kesalahan. Pilih satu spesimen uji yang **tidak dipakai saat penalaan**, lalu tampilkan kurva estimasi semua metode dan kesalahan posisinya. Garis merah bukan hasil anotasi manual; ia merupakan posisi batas dari generator fisika sintetis.

Perhatikan bagaimana estimasi bisa gagal pada baris tertentu, meskipun rata-rata baik. Hal ini lebih informatif untuk perbaikan instrumentasi daripada hanya mengoptimalkan satu angka. Dalam data nyata, kualitas *ground truth* dan variasi antar-anotator harus dievaluasi tersendiri.

In [ ]:
ex=test[4]
fig,axs=plt.subplots(1,2,figsize=(12.5,4.6))
tampil(axs[0],ex['image'],'Citra uji: overlay posisi batas')
axs[0].plot(ex['edge'],np.arange(HEIGHT),color='yellow',lw=2,label='Batas benar model')
for method,conf in selected.items():
    pos,_=estimasi_batas(ex['image'],method,**conf)
    axs[0].plot(pos,np.arange(HEIGHT),lw=1.0,alpha=.7,label=method)
    axs[1].plot(np.arange(HEIGHT),(pos-ex['edge'])*PITCH_MM,label=method,lw=1)
axs[0].set(xlim=(53,95),ylim=(HEIGHT,0));axs[0].legend(fontsize=7,loc='lower left',ncol=2)
axs[1].axhline(0,color='k',lw=1,ls='--')
axs[1].set(xlabel='Baris (piksel)',ylabel='Galat x (mm)',
           title='Residual posisi terhadap batas model')
axs[1].legend(fontsize=8);axs[1].grid(alpha=.2)
plt.tight_layout();plt.show()

## 18. Ketelitian fisik, ketidakpastian pitch, dan model pengukuran

Jika posisi tepi dipakai untuk menentukan perpindahan relatif terhadap garis acuan fixture $n_\mathrm{ref}$, besaran panjangnya adalah

$$
d=p\big(\widehat n-n_\mathrm{ref}\big),
$$

dengan pitch efektif $p$ dalam mm/piksel. Untuk pitch dan posisi relatif yang diasumsikan independen, pendekatan orde pertama memberikan

$$
u_d^2\approx(\widehat n-n_\mathrm{ref})^2u_p^2+p^2u_{\Delta n}^2.
$$

Jika ada kovarians, harus ditambahkan $2p(\widehat n-n_\mathrm{ref})\,\mathrm{Cov}(p,\Delta n)$. Pada sistem optik yang nyata, distorsi lensa, magnifikasi yang berubah terhadap kedalaman, dan orientasi objek dapat membuat satu pitch global tidak memadai.

**Peringatan inferensi:** simpangan baku antarbaris pada satu spesimen tidak otomatis mengukur ketidakpastian posisi rata-ratanya: bentuk batas yang memang bergelombang adalah variasi *measurand*, bukan noise pengukuran. Dibutuhkan desain pengulangan yang memisahkan variasi benda, alat, operator, dan posisi.

In [ ]:
# Ilustrasi gabungan komponen ketidakpastian; bukan hasil kalibrasi sensor
ref_col=72.0
meas_px=4.25   # contoh perpindahan posisi yang dibaca algoritma (piksel)
u_position_px=.36  # ilustrasi ketidakpastian standar estimasi posisi
pitch_mm=.08
u_pitch_mm=.0016  # 2% pitch; hanya asumsi demonstrasi
out_mm=pitch_mm*meas_px
u_total_mm=np.sqrt(meas_px**2*u_pitch_mm**2 + pitch_mm**2*u_position_px**2)
print(f'Perpindahan ilustratif: {out_mm:.4f} mm')
print(f'Ketidakpastian standar gabungan (asumsi independen): {u_total_mm:.4f} mm')
print('Kontribusi pitch:',round(meas_px**2*u_pitch_mm**2/u_total_mm**2,3),
      '| kontribusi posisi:',round(pitch_mm**2*u_position_px**2/u_total_mm**2,3))

## 19. Efisiensi komputasi dan kesetaraan pembandingan

Kinerja komputasi termasuk aspek desain algoritma: operator Sobel atau Scharr berbasis kernel kecil biasanya murah; Gaussian berskala besar meningkatkan biaya; dan Canny menambahkan NMS serta hysteresis. Namun biaya nyata dipengaruhi implementasi C/NumPy, ukuran citra, perangkat, memori, dan overhead pengukuran.

Ukur waktu pada **citra dan mesin yang sama**, dengan pemanasan awal, beberapa pengulangan, serta konfigurasi final yang dipilih dari validasi. Ini hanya *microbenchmark* lokal, bukan perkiraan kinerja produksi untuk semua perangkat atau resolusi.

In [ ]:
# Pengukuran runtuh waktu lokal (bukan benchmark lintas perangkat)
benchmark_image=test[0]['image']
bench=[]
for method,params in selected.items():
    for _ in range(2): estimasi_batas(benchmark_image,method,**params)
    times=[]
    for _ in range(6):
        t0=time.perf_counter()
        estimasi_batas(benchmark_image,method,**params)
        times.append((time.perf_counter()-t0)*1000)
    bench.append({'Metode':method,'Waktu median (ms)':np.median(times),
                  'Waktu min (ms)':np.min(times)})
bench_df=pd.DataFrame(bench).sort_values('Waktu median (ms)')
display(bench_df.round(3))
fig,ax=plt.subplots(figsize=(8.5,3))
ax.barh(bench_df['Metode'],bench_df['Waktu median (ms)'])
ax.invert_yaxis();ax.set(xlabel='Median waktu lokal per citra (ms)',
                        title='Biaya komputasi pada satu lingkungan pengujian')
ax.grid(alpha=.2,axis='x');plt.tight_layout();plt.show()

## 20. Verifikasi numerik dan audit reproducibility

Sebelum membandingkan metode, lakukan pemeriksaan yang dapat dijalankan ulang:

- Kernel turunan menjumlah nol (konstanta harus menghasilkan gradien nol di interior).
- Kernel ternormalisasi menghasilkan respons satu untuk bidang dengan kemiringan satu kode/piksel.
- Respons turunan Gaussian pada transisi naik berarah positif di interior.
- Pemisahan identitas dan *random seed* antara validasi dan pengujian.
- Seluruh metrik merupakan nilai hingga atau *NaN* yang memang ditandai sebagai tidak terdeteksi.
- Tidak ada fungsi estimator yang membaca koordinat *ground truth*; label hanya digunakan oleh fungsi evaluasi.
- Toleransi, ROI, penalti *missing*, skala geometris, dan kriteria seleksi ditetapkan sebelum evaluasi.

Lulus pengujian internal **tidak** menggantikan validasi instrumentasi, validasi perangkat, atau pemeriksaan pustaka setelah pembaruan versi.

In [ ]:
# Pemeriksaan terintegrasi; gagal keras jika ada kontrak numerik yang dilanggar
for name,K in operators.items():
    assert abs(K.sum())<1e-12, name
    assert np.allclose(ndimage.correlate(plane,K,mode='reflect')[6:-6,6:-6],1,atol=1e-12),name
assert len(validation)==9 and len(test)==13
assert {s['seed'] for s in validation}.isdisjoint({s['seed'] for s in test})
assert all(np.isfinite(s['image']).all() and s['image'].min()>=0 and s['image'].max()<=1
           for s in validation+test)
assert all(np.all((s['edge']>ROI_LO+2)&(s['edge']<ROI_HI-2)) for s in validation+test)
assert (df.mae_penalized_px >= 0).all()
assert (df.coverage.between(0,1)).all()
assert all(method in selected for method in methods)
print('SELURUH PEMERIKSAAN NUMERIK INTERNAL LULUS')

## 21. Audit asumsi dan batas interpretasi

Sebelum menyatakan suatu metode unggul, periksa hal berikut:

1. **Definisi tepi:** apakah yang dicari batas geometri, tepi intensitas, tepi bayangan, atau lokasi maksimum gradien?
2. **Konvensi koordinat:** apakah sistem memakai baris–kolom, $x,y$, orientasi sumbu, pitch, dan transformasi distorsi yang konsisten?
3. **Model sensor:** apakah citra linear terhadap irradiance? Bagaimana pengaruh gamma, *clipping*, debayering, dan noise berkorelasi?
4. **PSF:** apakah lebar/simetri PSF stabil terhadap lokasi, panjang gelombang, fokus, dan bidang kedalaman?
5. **Anotasi:** apakah posisi benar diketahui dengan metode terkalibrasi? Apakah penilai berbeda menghasilkan posisi yang sama?
6. **Protokol statistik:** apakah unit uji berbeda benar-benar independen pada tingkat yang relevan? Apakah parameter tidak diubah sesudah melihat uji?
7. **Kriteria performa:** apakah tujuan posisi, kontur utuh, jumlah objek, atau segmentasi? Apakah metrik relevan terhadap tujuan?
8. **Ambang keputusan:** bagaimana biaya kegagalan deteksi dibanding bias lokalisasi? Apakah toleransi secara fisik bermakna?
9. **Generalisasi:** apakah metode bertahan pada pencahayaan, material, sensor, resolusi, atau protokol akuisisi berbeda?
10. **Batas klaim:** apakah hasil adalah bukti untuk satu simulator, satu dataset, atau beberapa domain dunia nyata?

**Kesimpulan prinsipil:** peningkatan *edge contrast*, jumlah piksel kontur, atau visualisasi kontur yang tipis **bukan** bukti langsung bahwa posisi batas fisik terukur semakin akurat.

## 22. Latihan bertingkat, HOTS, dan proyek mini

**Latihan 1 — Beda hingga (dasar).** Diberikan profil $f=[0,0,0.1,0.4,0.8,0.9,0.9]$ dan $\Delta x=0.2$ mm. Hitung turunan tengah pada indeks 2, 3, dan 4; jelaskan satuan hasilnya dan mengapa hasil tidak dapat dibandingkan dengan turunan kode/piksel tanpa konversi.

**Latihan 2 — Kernel (dasar).** Tunjukkan bahwa $K_{\mathrm{Sobel},x}$ mempunyai jumlah bobot nol. Uji pada citra konstan dan bidang miring $f[m,n]=c+an+bm$; jelaskan mengapa transposenya diperlukan.

**Latihan 3 — Analisis spektral (menengah).** Turunkan $D(\omega)=i\sin\omega$ untuk beda tengah dengan $\Delta x=1$. Jelaskan mengapa operator menghilangkan respons tepat pada frekuensi Nyquist dan konsekuensinya untuk pola garis yang berselang satu piksel.

**Latihan 4 — ESF/MTF (menengah).** Pada model Gaussian dengan $\sigma=1.6$ piksel, hitung MTF pada $\nu=0.1$ dan $0.25$ siklus/piksel. Bagaimana hasil berubah ketika lebar PSF meningkat 50%? Bedakan efek blur instrumen dan penghalusan digital tambahan.

**Latihan 5 — Canny (menengah).** Pada citra yang sama, ubah $\sigma$, ambang rendah, dan ambang tinggi. Laporkan jumlah tepi, keterhubungan, serta dua contoh tepi palsu dan tepi terlewat. Jelaskan mengapa pengurangan jumlah piksel tepi bukan ukuran kualitas yang cukup.

**Latihan 6 — Subpiksel dan bias (lanjutan).** Bandingkan estimator integer, parabola, dan *fit* ESF Gaussian pada 200 pengulangan transisi sintetis. Ubah PSF menjadi asimetris dan uji apakah ketiga metode masih tak bias. Lakukan analisis sensitivitas ketika kontras menurun.

**Latihan 7 — Evaluasi tanpa leakage (HOTS).** Buat dua domain pencitraan: domain A memiliki $\sigma_{\mathrm{PSF}}\in[0.8,1.5]$, domain B $\sigma_{\mathrm{PSF}}\in[2.0,3.2]$. Pilih parameter hanya pada A, uji pada B, laporkan degradasi MAE dan coverage, lalu usulkan desain eksperimen yang dapat mengidentifikasi penyebab penurunan.

**Latihan 8 — Metrologi dan ketidakpastian (HOTS).** Rancang pengukuran posisi tepi untuk kamera 1920×1080 dengan objek pada dua bidang kedalaman. Susun persamaan pengukuran, strategi kalibrasi pitch/distorsi, ketidakpastian posisi, dan standar referensi. Jelaskan mengapa ulangan baris tidak otomatis sama dengan ulangan alat.

**Proyek mini (kelompok 2–3 orang):** pilih satu sistem tepi yang memiliki fungsi ukur nyata, misalnya tepi lembaran, garis meniskus, atau batas deposisi. Definisikan measurand, buat/akses dataset dengan lisensi sah, sediakan *ground truth* yang defensibel, bandingkan minimal tiga metode, pisahkan validasi–uji pada tingkat spesimen, dan tulis *technical report*: masalah → model → metode → protokol → hasil → ketidakpastian → kegagalan → batas klaim. Jangan gunakan citra sintetis sebagai bukti validasi lapangan.

## 23. Rangkuman dan koneksi berikutnya

1. **Histogram** menjelaskan distribusi intensitas; deteksi tepi memanfaatkan hubungan spasial untuk menandai perubahan sinyal yang terlokalisasi.
2. **Gradien** adalah turunan spasial yang tergantung grid, skala, noise, dan konvensi koordinat. Besarnya tidak langsung menjadi ukuran akurasi geometris.
3. **Sobel, Scharr, dan Canny** menerapkan asumsi serta kompromi berbeda antara deteksi, lokalisasi, dan respons tunggal. Keluaran Canny biner tidak otomatis memberi koordinat subpiksel.
4. **PSF–ESF–LSF–MTF** menjelaskan bagaimana transisi objek dibentuk oleh sistem pencitraan dan bagaimana blur mengubah profil serta kemampuan lokalisasi.
5. **Pengukuran ilmiah** memerlukan *ground truth* defensibel, kalibrasi skala, estimasi bias dan presisi, serta protokol evaluasi yang mencegah *data leakage*.
6. **Morfologi dan segmentasi** mengubah deteksi perubahan intensitas menjadi hipotesis wilayah/objek; jumlah komponen digital tidak selalu sama dengan jumlah benda fisik.

**Koneksi berikutnya:** Notebook 05 memperdalam segmentasi, morfologi, dan keterhubungan objek; analisis domain frekuensi menelusuri respons kernel dan MTF lebih jauh; *inverse problems* mencoba membalikkan efek blur/noise dengan regularisasi; dan pengukuran ilmiah membahas ketidakpastian kalibrasi serta validasi geometri. Semua materi itu melanjutkan gagasan utama dari 01–04: **citra merupakan keluaran sistem ukur, bukan sekadar kumpulan piksel untuk diproses.**

## 24. Referensi, dokumentasi perangkat lunak, dan provenance

- Canny, J. (1986). “A Computational Approach to Edge Detection.” *IEEE Transactions on Pattern Analysis and Machine Intelligence*, PAMI-8(6), 679–698. DOI: https://doi.org/10.1109/TPAMI.1986.4767851.
- Gonzalez, R. C., & Woods, R. E. (2018). *Digital Image Processing* (edisi ke-4). Pearson. Bab terkait transformasi spasial dan segmentasi.
- Szeliski, R. (2022). *Computer Vision: Algorithms and Applications* (edisi ke-2). Springer. https://szeliski.org/Book/.
- Lindeberg, T. (1994). *Scale-Space Theory in Computer Vision*. Springer. https://doi.org/10.1007/978-1-4757-6465-9.
- ISO 12233 (rujukan topik): *Photography — Electronic still picture imaging — Resolution and spatial frequency responses*; gunakan versi standar yang berlaku melalui akses resmi, bukan asumsi bahwa ilustrasi notebook merupakan pengujian sertifikasi ISO.
- `scikit-image` API — Sobel dan Scharr: https://scikit-image.org/docs/stable/api/skimage.filters.html.
- `scikit-image` API — Canny: https://scikit-image.org/docs/stable/api/skimage.feature.html#skimage.feature.canny.
- `SciPy ndimage`: https://docs.scipy.org/doc/scipy/reference/ndimage.html.

**Keterangan provenance:** citra `camera` dan `coins` berasal dari contoh `scikit-image`; eksperimen lain dibangun dari fungsi matematika dan *random seed* yang tercantum di notebook. Foto alat, citra pengujian lapangan, data sensor terkalibrasi, serta label anotasi manusia **tidak tersedia** dalam notebook ini. Angka evaluasi tidak boleh ditafsirkan sebagai performa sistem nyata.

**Pemeriksaan ilmiah yang dianjurkan:** uji pada lingkungan paket berbeda (termasuk Colab), ukur waktu dan akurasi sensor nyata, tetapkan *calibration protocol*, lakukan evaluasi lintas domain, dan dokumentasikan keterbatasan sebelum klaim generalisasi.